In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


c:\Users\paras\OneDrive\Desktop\3_project_codebasics_q_and_a\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load API key from .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")


llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=api_key,
    temperature=0.1
)

In [3]:
# Testing simple generation
poem_response = llm.invoke("Write a 4 line poem of my love for samosa")
print(poem_response.content)


[{'type': 'text', 'text': 'Oh, golden triangle of crispy grace,\nNo other snack could ever take your place.\nWith spicy warmth and chutney on the side,\nIn every bite, my heart is satisfied.', 'extras': {'signature': 'EvARCu0RAQw51scZzptZyoeskyzdzHj9+dflS2qjx7aLRFVIwdXuSbmVxavKthWxr/zToiVPyR7hEnmG1nwaTghF3X1Iiu0XdN00tbfebMVGOf7lyPPa00ZHI4dho4GnWxmy7iX7Q+yypAWXVEJT2APFX8Qrgf60q76z5DTX0xIupv94iYLjp2lCjTg57mpdarSHgTOO7eRiwlNRmqrzeJGKva9CKWPglI4+jd3PtTA5OXH+44xSYzUhnBkdBf4U34MtwpEqibHU9V6hjdnTD60JK34KfobBKdlgX84NBPMUQXUw03XR47Ld5k2A9EhN2FgtFkRYGMMihdgmzT32OnipEJaN5b6u+KmQ1jGwXPu6WC/HZGBkPmOheMpazJmxBcHz5Or3OzjKdvMtIEa1Ym7MA4naYqdWiRUs9Or7j9bYV0ZZWiJTWWE9CsQLWXbOKbtUCY1GRMbZCRRHxXYP9g9TugAnjJ2ry3M37cYs6OnB7QUVcZtKu/05NrmdHoL53iqiVXWN7XsrYTIqzKODIGBSa1llYTVxCIjNAF26URg8KfvkKB7DlGY4bJWSiALgZQz9Jc2HRaC83XgNXDIJMCBzOHqwqASQM7iHH+f8ljBndjZ+PaA/TRRjNsTy7LYtXJ9a8tfJlidbCYskVU8ac03mAuh4pIcnuMUv3Pu+ET5BqDeaYo+4+iLC38Vgqq0rVwEdNTnNzxI8Iu6z513itMB9Fvu9ZCcJEcSkjAwc0lE8Ep2R+M1Ztuus5dlfz3BJVbRsrYQ5G3bfnu

In [6]:
from langchain_community.document_loaders.csv_loader import CSVLoader
import os

csv_path = os.path.join("dataset", "codebasics_faqs.csv")  # or "data", "qa_faqs.csv" based on step 2

print("Using csv_path:", csv_path)
print("Exists?", os.path.exists(csv_path))

loader = CSVLoader(
    file_path=csv_path,
    source_column="prompt",
    encoding="latin-1",  # keep if needed
)

data = loader.load()
print(f"Loaded {len(data)} documents")
data[0]

Using csv_path: dataset\codebasics_faqs.csv
Exists? True
Loaded 75 documents


Document(metadata={'source': 'I have never done programming in my life. Can I take this bootcamp?', 'row': 0}, page_content='prompt: I have never done programming in my life. Can I take this bootcamp?\nresponse: Yes, this is the perfect bootcamp for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in your current job or business using data.')

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

# You can keep using the same model; it will work without explicit query_instruction.
embeddings = HuggingFaceEmbeddings(
    model_name="hkunlp/instructor-large"
)

# Test embedding query
e = embeddings.embed_query("What is your refund policy?")
print("Embedding successful, dim:", len(e))
print(e[:5])

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 17633.71it/s]


Embedding successful, dim: 768
[-0.04449571296572685, 0.007691540755331516, -0.009869123809039593, 0.020831892266869545, 0.031858958303928375]


In [10]:
from langchain_community.vectorstores import FAISS

# Create FAISS instance (use the new embeddings object)
vectordb = FAISS.from_documents(
    documents=data,
    embedding=embeddings
)

# Create a retriever
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test retriever
rdocs = retriever.invoke("how about job placement support?")

# Verification check
if not rdocs:
    print("No documents found. Check if the 'data' variable is empty.")
else:
    print(f"Found {len(rdocs)} documents:")
    print("-" * 30)
    for doc in rdocs:
        print(doc.page_content)
        print("-" * 10)

Found 3 documents:
------------------------------
prompt: Do you provide any job assistance?
response: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.
----------
prompt: Do you provide any virtual internship?
response: Yes
----------
prompt: Will this bootcamp guarantee me a job?
response: The courses included in this bootcamp are done by 9000+ learners and many of them have secured a job which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.
----------


In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from the "response" section in the source document context without making many changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT:
{context}

QUESTION:
{question}
""",
    input_variables=["context", "question"],
)

In [15]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# build {context, question, source_documents}
setup_and_retrieval = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "source_documents": retriever,
    }
)

rag_chain_with_sources = (
    setup_and_retrieval
    | {
        "result": prompt | llm | StrOutputParser(),
        "source_documents": lambda x: x["source_documents"],
    }
)

In [17]:
# 4. Execute the query
query = "I’m not sure if this course is good enough for me to invest some money. What can I do?"
response = rag_chain_with_sources.invoke(query)

print("Final Answer:")
print(response["result"])

print("\n" + "-"*30)
print("Source Documents Matching:")
for i, doc in enumerate(response["source_documents"]):
    print(f"\nDocument {i+1}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

Final Answer:
Don’t worry. Many videos in this course are free so watch them to get an idea of the quality of teaching. Dhaval Patel (the course instructor) runs a popular data science YouTube channel called Codebasics. On that, you can watch his videos and read comments to get an idea of his teaching style

------------------------------
Source Documents Matching:

Document 1:
Content: prompt: Im not sure if this course is good enough for me to invest some money. What can I do?
response: Dont worry. Many videos in this course are free so watch them to get an idea of the quality of teaching. Dhaval Patel (the course instructor) runs a popular data science YouTube channel called Codebasics. On that, you can watch his videos and read comments to get an idea of his teaching style
Metadata: {'source': 'I\x92m not sure if this course is good enough for me to invest some money. What can I do?', 'row': 20}

Document 2:
Content: prompt: Im not sure if this bootcamp is good enough for me to 